# 07b — ABM Validation: Proposed Network Utilization & AFIR Compliance

**Validates the proposed station network (NB07 output) using the calibrated ABM.**

For each proposed station: computes average-day and peak-day utilization, queue risk,
and stranded-demand estimate. Then re-runs the AFIR gap detector against the combined
(existing + proposed) network to confirm full compliance.

Inputs:
- `data/processed/proposed_stations.csv` — NB07 output
- `data/processed/demand_per_segment.csv` — NB06 ABM demand
- `data/processed/interurban_roads.parquet` — road network
- `data/processed/interurban_chargers_baseline.csv` — existing charger baseline

Output:
- `data/processed/station_validation_metrics.csv`
- `data/processed/fig_07b_utilization_validation.png`

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
    DATA_DIR = Path('../data/processed')
else:
    sys.path.insert(0, os.getcwd())
    DATA_DIR = Path('data/processed')

from src.constants import (
    CHARGING_PROBABILITY,
    AVG_CHARGE_DURATION_HOURS,
    EFFECTIVE_OPERATING_HOURS,
    MIN_EXISTING_CHARGER_POWER_KW,
    POWER_PER_CHARGER_KW,
)
from src.abm_demand import get_seasonal_multiplier
from src.optimization import compute_coverage_gaps
from src.data_loading import load_geo_parquet_compat

print('ABM parameters:')
print(f'  B1 (charging probability):    {CHARGING_PROBABILITY}')
print(f'  B2 (session duration):         {AVG_CHARGE_DURATION_HOURS} h ({AVG_CHARGE_DURATION_HOURS*60:.0f} min)')
print(f'  B3 (operating hours/day):      {EFFECTIVE_OPERATING_HOURS} h')
print(f'  Charger power:                 {POWER_PER_CHARGER_KW} kW')
print(f'  Capacity per charger per day:  {EFFECTIVE_OPERATING_HOURS / AVG_CHARGE_DURATION_HOURS:.0f} sessions')

## Step 1: Load Inputs

In [ ]:
# Proposed stations from NB07
proposed_path = DATA_DIR / 'proposed_stations.csv'
assert proposed_path.exists(), 'Run NB07 first to generate proposed_stations.csv'
proposed = pd.read_csv(proposed_path)
print(f'📍 Proposed stations:       {len(proposed)}')
print(f'   Total proposed chargers:  {proposed["n_chargers_proposed"].sum()}')
print(f'   Total station capacity:   {proposed["n_chargers_proposed"].sum() * POWER_PER_CHARGER_KW:,} kW')
print(f'   Routes: {sorted(proposed["route_segment"].tolist())}')

# ABM demand per segment from NB06
demand = pd.read_csv(DATA_DIR / 'demand_per_segment.csv')
print(f'\n📈 Demand segments:         {len(demand)}')
print(f'   Routes in demand data:    {demand["route_segment"].nunique()}')

# Road network (for AFIR re-check in Step 3)
roads = load_geo_parquet_compat(DATA_DIR / 'interurban_roads.parquet')

def _tent_tier(row):
    val = row.get('TENT_red_basica')
    if isinstance(val, str):
        v = val.strip().lower()
        if v == 'core': return 'core'
        if v == 'comprehensive': return 'comprehensive'
    return 'core' if row.get('is_tent', False) else 'none'

roads['tent_tier'] = roads.apply(_tent_tier, axis=1)
print(f'\n🛣️  Road segments:           {len(roads)}')

# Existing charger baseline (for combined-network AFIR check)
chargers = pd.read_csv(DATA_DIR / 'interurban_chargers_baseline.csv')
fast_chargers = chargers[chargers['max_power_kw'] >= MIN_EXISTING_CHARGER_POWER_KW]
print(f'🔌 Fast chargers baseline:   {len(fast_chargers)} (≥{MIN_EXISTING_CHARGER_POWER_KW} kW)')

## Step 2: Station-Level Utilization Metrics

For each proposed station:
1. Find all demand segments on the same route (from `demand_per_segment.csv`)
2. Compute length-weighted average daily BEV flow (representative of the gap location)
3. Apply the ABM seasonal multiplier for the peak-day scenario
4. Compute utilization, queue risk, and stranded-demand estimate

**Utilization** = `(daily_sessions × session_duration) / (n_chargers × operating_hours)`

**Queue risk thresholds** (from M/M/c queuing theory):
- Low: peak utilisation < 65% — smooth operation
- Moderate: 65–85% — occasional waits
- High: 85–100% — frequent queues likely
- Severe: > 100% — demand exceeds capacity (sizing error)

In [ ]:
# Sessions capacity per charger per day
CAPACITY_PER_CHARGER_PER_DAY = EFFECTIVE_OPERATING_HOURS / AVG_CHARGE_DURATION_HOURS

records = []
for _, sta in proposed.iterrows():
    route  = sta['route_segment']
    n_c    = int(sta['n_chargers_proposed'])

    # --- Demand context: segments on this route ---
    segs = demand[demand['route_segment'] == route]

    if len(segs) == 0:
        # No segment data for this route (shouldn't happen with complete pipeline)
        bev_avg = 0.0
        seasonal_mult = get_seasonal_multiplier(route, scenario='peak')
    else:
        # Length-weighted average BEV flow (average-day, no seasonal scaling)
        weights = segs['length_km'].values
        bev_avg = float(np.average(segs['daily_bev_traffic_2027'].values, weights=weights))
        # Peak seasonal multiplier from demand data (conservative: take max over segments)
        seasonal_mult = float(segs['seasonal_multiplier'].max())

    # Cross-check with ABM model function (use the larger of the two — conservative)
    seasonal_mult_abm = get_seasonal_multiplier(route, scenario='peak')
    seasonal_mult = max(seasonal_mult, seasonal_mult_abm)

    bev_peak = bev_avg * seasonal_mult

    # Daily charging sessions
    avg_sessions  = bev_avg  * CHARGING_PROBABILITY
    peak_sessions = bev_peak * CHARGING_PROBABILITY

    # Station capacity (sessions/day)
    capacity_sessions = n_c * CAPACITY_PER_CHARGER_PER_DAY

    # Utilization rates (fraction of total charger-hours consumed)
    avg_util  = (avg_sessions  * AVG_CHARGE_DURATION_HOURS) / (n_c * EFFECTIVE_OPERATING_HOURS)
    peak_util = (peak_sessions * AVG_CHARGE_DURATION_HOURS) / (n_c * EFFECTIVE_OPERATING_HOURS)

    # Queue risk classification
    if peak_util >= 1.0:
        queue_risk = 'Severe'
    elif peak_util >= 0.85:
        queue_risk = 'High'
    elif peak_util >= 0.65:
        queue_risk = 'Moderate'
    else:
        queue_risk = 'Low'

    # Stranded sessions at peak (demand that cannot be served if overloaded)
    stranded_peak = max(0.0, peak_sessions - capacity_sessions)

    records.append({
        'location_id':               sta['location_id'],
        'latitude':                  sta['latitude'],
        'longitude':                 sta['longitude'],
        'route_segment':             route,
        'n_chargers_proposed':       n_c,
        'station_power_kw':          n_c * POWER_PER_CHARGER_KW,
        'bev_flow_avg_day':          round(bev_avg, 1),
        'bev_flow_peak_day':         round(bev_peak, 1),
        'seasonal_multiplier':       seasonal_mult,
        'avg_sessions_per_day':      round(avg_sessions, 1),
        'peak_sessions_per_day':     round(peak_sessions, 1),
        'capacity_sessions_per_day': round(capacity_sessions, 1),
        'avg_utilization':           round(avg_util, 3),
        'peak_utilization':          round(peak_util, 3),
        'queue_risk':                queue_risk,
        'stranded_sessions_peak':    round(stranded_peak, 1),
    })

metrics = pd.DataFrame(records)

print('📊 Station-level utilization metrics:')
print(metrics[[
    'location_id', 'route_segment', 'n_chargers_proposed',
    'bev_flow_avg_day', 'seasonal_multiplier',
    'avg_utilization', 'peak_utilization', 'queue_risk'
]].to_string(index=False))
print(f'\n   Mean avg utilization:  {metrics["avg_utilization"].mean():.1%}')
print(f'   Mean peak utilization: {metrics["peak_utilization"].mean():.1%}')
print(f'\n   Queue risk distribution:')
for risk, cnt in metrics['queue_risk'].value_counts().items():
    print(f'     {risk}: {cnt}')

## Step 3: AFIR Compliance Re-Check (Post-Placement)

Re-run `compute_coverage_gaps` against the **combined** network (existing fast chargers + all proposed stations).
A correctly placed network should return 0 remaining gaps.

In [ ]:
# Build combined charger set: existing fast chargers + proposed stations
# Each proposed station row represents one location with ≥150 kW → counts as AFIR coverage
proposed_as_chargers = pd.DataFrame({
    'latitude':     proposed['latitude'],
    'longitude':    proposed['longitude'],
    'max_power_kw': float(POWER_PER_CHARGER_KW),  # one charger point per row ≥ 50 kW
})

combined_chargers = pd.concat([
    chargers[['latitude', 'longitude', 'max_power_kw']],
    proposed_as_chargers,
], ignore_index=True)

n_fast_combined = (combined_chargers['max_power_kw'] >= MIN_EXISTING_CHARGER_POWER_KW).sum()
print(f'Combined network: {len(combined_chargers):,} locations ({n_fast_combined:,} fast ≥{MIN_EXISTING_CHARGER_POWER_KW} kW)')
print(f'  — of which {len(proposed)} are newly proposed stations')

gaps_after = compute_coverage_gaps(
    road_segments_df=roads,
    existing_stations_df=combined_chargers,
)

print(f'\n🛡️  AFIR compliance after placement:')
if len(gaps_after) == 0:
    print('   ✅ 0 uncovered stretches — all AFIR spacing requirements met.')
else:
    print(f'   ⚠️  {len(gaps_after)} gap(s) remain after placement:')
    print(gaps_after[['Carretera', 'gap_length_km', 'gap_spacing_threshold_km',
                       'is_tent', 'n_chargers_on_route']].to_string(index=False))

## Step 4: Summary KPIs & Export

In [ ]:
# --- Aggregate KPIs ---
total_stations     = len(proposed)
total_chargers     = int(proposed['n_chargers_proposed'].sum())
total_capacity_kw  = total_chargers * POWER_PER_CHARGER_KW
avg_util_mean      = metrics['avg_utilization'].mean()
peak_util_mean     = metrics['peak_utilization'].mean()
overloaded_peak    = int((metrics['peak_utilization'] >= 1.0).sum())
high_queue_risk    = int(metrics['queue_risk'].isin(['High', 'Severe']).sum())
total_stranded     = float(metrics['stranded_sessions_peak'].sum())
afir_gaps_remain   = len(gaps_after)

print('=' * 60)
print('NB07b — ABM VALIDATION SUMMARY')
print('=' * 60)
print(f'  Proposed stations:             {total_stations}')
print(f'  Total proposed chargers:       {total_chargers}')
print(f'  Total station capacity:        {total_capacity_kw:,} kW')
print(f'  Mean utilization (avg-day):    {avg_util_mean:.1%}')
print(f'  Mean utilization (peak-day):   {peak_util_mean:.1%}')
print(f'  Stations overloaded at peak:   {overloaded_peak}')
print(f'  Stations at high queue risk:   {high_queue_risk}')
print(f'  Stranded sessions (peak/day):  {total_stranded:.0f}')
print(f'  Post-placement AFIR gaps:      {afir_gaps_remain}')
print('=' * 60)

# Sizing sanity check: no station should be overloaded at average-day demand
# (stations were sized for peak; average-day utilization must always be < 1)
overloaded_avg = int((metrics['avg_utilization'] >= 1.0).sum())
assert overloaded_avg == 0, (
    f'{overloaded_avg} stations overloaded at average demand — '
    'indicates a mismatch between NB06 demand sizing and NB07 placement'
)
print('✅ Sizing check passed: no stations overloaded at average-day demand')

# Export
out_path = DATA_DIR / 'station_validation_metrics.csv'
metrics.to_csv(out_path, index=False)
print(f'💾 Saved → {out_path}')
metrics

## Step 5: Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Left: utilization distribution (avg vs peak) ---
ax = axes[0]
max_util = max(metrics['peak_utilization'].max(), 1.05)
bins = np.linspace(0, max_util, 15)
ax.hist(metrics['avg_utilization'],  bins=bins, alpha=0.7, label='Avg-day',  color='steelblue')
ax.hist(metrics['peak_utilization'], bins=bins, alpha=0.7, label='Peak-day', color='tomato')
ax.axvline(0.65, color='orange',  linestyle='--', linewidth=1,   label='Moderate risk (65%)')
ax.axvline(0.85, color='red',     linestyle='--', linewidth=1,   label='High risk (85%)')
ax.axvline(1.00, color='darkred', linestyle='-',  linewidth=1.5, label='Overloaded (100%)')
ax.set_xlabel('Utilization rate')
ax.set_ylabel('Number of stations')
ax.set_title('Station Utilization Distribution')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend(fontsize=8)

# --- Right: charger count vs peak utilization, coloured by queue risk ---
ax = axes[1]
risk_colors = {'Low': '#2ca02c', 'Moderate': '#ff7f0e', 'High': '#d62728', 'Severe': '#7b0000'}
for risk, grp in metrics.groupby('queue_risk'):
    ax.scatter(
        grp['n_chargers_proposed'], grp['peak_utilization'],
        color=risk_colors.get(risk, 'grey'), label=risk, s=90, zorder=3
    )
    for _, row in grp.iterrows():
        ax.annotate(
            row['route_segment'],
            (row['n_chargers_proposed'], row['peak_utilization']),
            fontsize=7, ha='left', va='bottom',
            xytext=(4, 3), textcoords='offset points'
        )
ax.axhline(0.85, color='red',     linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(1.00, color='darkred', linestyle='-',  linewidth=1, alpha=0.7)
ax.set_xlabel('Chargers proposed per station')
ax.set_ylabel('Peak utilization rate')
ax.set_title('Peak Utilization by Charger Count')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend(title='Queue risk', fontsize=8)

plt.suptitle(
    'NB07b — Proposed Network: ABM Utilization Validation',
    fontsize=12, y=1.02
)
plt.tight_layout()

fig_path = DATA_DIR / 'fig_07b_utilization_validation.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Saved → {fig_path}')